# 3. Confidence Calibration

This notebook demonstrates how `confidence_scorer.py` works. We simulate a batch of 10 validation results, compute composite confidence scores using weighted components, calibrate against a threshold of 0.70, and visualise the distribution.

**Score components & weights:**
- Spectral contrast weight: 0.40
- Movement (strain) weight: 0.30
- Deviation weight: 0.30

## Imports & setup

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), "..", "backend"))

from edge_engine.confidence_scorer import (
    ConfidenceWeights,
    compute_confidence,
    is_ambiguous,
)

print("All imports successful.")

## 3.1 Simulate 10 validation results

Each result corresponds to a single polygon boundary check. We generate plausible values for:
- `spectral_contrast` — clarity of edge in pixel space (0.0–1.0)
- `total_vertex_movement` — total snake displacement (pixels)
- `polygon_perimeter` — length of the ML-predicted boundary (pixels)
- `mean_deviation` — average per-vertex deviation (pixels)

In [ ]:
np.random.seed(123)
n_results = 10

spectral_contrasts = np.random.beta(3, 2, size=n_results)  # mostly 0.4–0.9
total_movements = np.random.exponential(scale=15, size=n_results)  # pixels
perimeters = np.random.uniform(80, 200, size=n_results)  # pixels
mean_deviations = np.random.exponential(scale=1.5, size=n_results)  # pixels

print("Simulated validation results:")
print(f"{'#':>2}  {'Sp. Contrast':>12}  {'Movement':>8}  {'Perimeter':>9}  {'Deviation':>9}")
print("-" * 46)
for i in range(n_results):
    print(
        f"{i:>2}  {spectral_contrasts[i]:>12.4f}  {total_movements[i]:>8.2f}  {perimeters[i]:>9.2f}  {mean_deviations[i]:>9.4f}"
    )

## 3.2 Compute confidence scores

We use the default `ConfidenceWeights(spectral=0.4, strain=0.3, deviation=0.3)` and a buffer width of 2.0 metres, matching the production config.

In [ ]:
weights = ConfidenceWeights(spectral=0.40, strain=0.30, deviation=0.30)
threshold = 0.70
buffer_width = 2.0

confidences = []
ambiguities = []
for i in range(n_results):
    conf = compute_confidence(
        spectral_contrast=spectral_contrasts[i],
        total_vertex_movement=total_movements[i],
        polygon_perimeter=perimeters[i],
        mean_deviation=mean_deviations[i],
        buffer_width_meters=buffer_width,
        weights=weights,
    )
    ambig = is_ambiguous(conf, threshold)
    confidences.append(conf)
    ambiguities.append(ambig)

confidences = np.array(confidences)
print(f"{'#':>2}  {'Confidence':>10}  {'Ambiguous?':>10}")
print("-" * 26)
for i in range(n_results):
    flag = "YES" if ambiguities[i] else "no"
    print(f"{i:>2}  {confidences[i]:>10.4f}  {flag:>10}")

## 3.3 Component breakdown

Let's look at how each term contributes to the final score for the first result.

In [ ]:
i = 0
sp = spectral_contrasts[i]
mov = total_movements[i]
perim = perimeters[i]
dev = mean_deviations[i]

snake_strain = min(mov / max(perim, 1e-6), 1.0)
norm_dev = min(dev / max(buffer_width, 1e-6), 1.0)

term_spec = weights.spectral * sp
term_strain = weights.strain * (1.0 - snake_strain)
term_dev = weights.deviation * (1.0 - norm_dev)

print(f"Result #{i} break-down:")
print(f"  spectral_contrast          = {sp:.4f}  × w{weights.spectral} = {term_spec:.4f}")
print(
    f"  1 - snake_strain          = {1 - snake_strain:.4f}  × w{weights.strain} = {term_strain:.4f}   (strain = {snake_strain:.4f})"
)
print(
    f"  1 - normalized_deviation  = {1 - norm_dev:.4f}  × w{weights.deviation} = {term_dev:.4f}   (norm_dev = {norm_dev:.4f})"
)
print("  ─────────────────────────────────────")
print(f"  Confidence score          = {confidences[i]:.4f}")

## 3.4 Calibrate the threshold

We test a sweep of candidate thresholds from 0.50 to 0.90 and count how many results are flagged as ambiguous.

In [ ]:
candidate_thresholds = np.linspace(0.50, 0.90, 9)
print(f"{'Threshold':>10}  {'Ambiguous count':>16}")
print("-" * 28)
for t in candidate_thresholds:
    cnt = sum(is_ambiguous(c, t) for c in confidences)
    print(f"{t:>10.2f}  {cnt:>16}")

## 3.5 Plot confidence distribution

We visualise the distribution of computed confidences and mark the 0.70 threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
colors = ["crimson" if a else "seagreen" for a in ambiguities]
axes[0].bar(range(n_results), confidences, color=colors, edgecolor="black")
axes[0].axhline(
    threshold, color="gray", linestyle="--", linewidth=1.5, label=f"Threshold = {threshold}"
)
axes[0].set_xlabel("Validation result #")
axes[0].set_ylabel("Confidence score")
axes[0].set_title("Per-result confidence scores")
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# Histogram
axes[1].hist(confidences, bins=8, color="steelblue", edgecolor="black", alpha=0.7)
axes[1].axvline(
    threshold, color="gray", linestyle="--", linewidth=1.5, label=f"Threshold = {threshold}"
)
axes[1].set_xlabel("Confidence score")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Confidence distribution")
axes[1].legend()

fig.tight_layout()
plt.show()

ambiguous_count = sum(ambiguities)
print(f"Total ambiguous results (confidence < {threshold}): {ambiguous_count} / {n_results}")

## 3.6 How `confidence_scorer.py` works

1. **`ConfidenceWeights`** — a simple `dataclass` with default weights `(0.4, 0.3, 0.3)`.
2. **`compute_confidence`** — 
   - `snake_strain` = total_vertex_movement / polygon_perimeter (capped at 1.0)
   - `normalized_deviation` = mean_deviation / buffer_width_meters (capped at 1.0)
   - Score = `w_spec * spec + w_strain * (1 - strain) + w_dev * (1 - norm_dev)`
   - Clamped to [0, 1] and rounded to 4 decimals.
3. **`is_ambiguous`** — returns `True` when `confidence < threshold` (default 0.70).

In [ ]:
print("confidence_scorer.py source:")
with open("../backend/edge_engine/confidence_scorer.py") as f:
    print(f.read())